In [2]:
import pandas as pd
import numpy as np

# =========================================================
# PART A — DATA UNDERSTANDING
# =========================================================

# 1. Load the dataset

df = pd.read_csv('C:/Users/md.shamim/source/IITM/StudyMaterial/Python_Assignment/Day4/ecommerce_orders_raw.csv')

print("Dataset loaded successfully")


# 2. Display structure and dimensions

print("\nFirst 5 records:")
print(df.head())

print("\nDataset dimensions:")
print(df.shape)

print("\nDataset information:")
print(df.info())


# 3. Data types of all columns

print("\nData Types:")
print(df.dtypes)


# 4. Identify missing values

print("\nMissing Values:")
print(df.isnull().sum())


# 5. Identify duplicate orders

print("\nDuplicate records:")
print(df.duplicated().sum())

print("\nDuplicate Order IDs:")
print(df["Order_ID"].duplicated().sum())


# 6. Examine unique values

print("\nUnique Cities:")
print(df["City"].unique())

print("\nUnique Product Categories:")
print(df["Product_Category"].unique())

print("\nUnique Payment Methods:")
print(df["Payment_Method"].unique())


# =========================================================
# PART B — DATA CLEANING
# =========================================================

# 7. Handle missing ratings

print("\nMissing ratings before cleaning:")
print(df["Rating"].isnull().sum())

# Use median rating
df["Rating"] = df["Rating"].fillna(df["Rating"].median())

print("Missing ratings after cleaning:")
print(df["Rating"].isnull().sum())


# 8. Standardize city names

print("\nCities before standardization:")
print(df["City"].value_counts())

df["City"] = (
    df["City"]
    .str.strip()
    .str.title()
)

print("\nCities after standardization:")
print(df["City"].value_counts())


# 9. Identify invalid quantities

invalid_quantity = df[df["Quantity"] <= 0]

print("\nInvalid quantity records:")
print(invalid_quantity)

print("\nNumber of invalid quantities:")
print(len(invalid_quantity))


# Handle invalid quantities
# Since quantity cannot be zero or negative,
# remove those records.

df = df[df["Quantity"] > 0].copy()


# 10. Handle duplicate orders

print("\nDuplicate rows before removal:")
print(df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after removal:")
print(df.duplicated().sum())


# 11. Convert Order_Date to datetime

df["Order_Date"] = pd.to_datetime(df["Order_Date"])

print("\nOrder Date datatype:")
print(df["Order_Date"].dtype)


# =========================================================
# PART C — FEATURE ENGINEERING
# =========================================================

# 12. Gross Amount

df["Gross_Amount"] = (
    df["Quantity"] * df["Unit_Price"]
)


# 13. Discount Amount

df["Discount_Amount"] = (
    df["Gross_Amount"] *
    df["Discount"] / 100
)


# 14. Net Amount

df["Net_Amount"] = (
    df["Gross_Amount"] -
    df["Discount_Amount"]
)


# Display calculated columns

print("\nAmount calculations:")
print(
    df[
        [
            "Quantity",
            "Unit_Price",
            "Discount",
            "Gross_Amount",
            "Discount_Amount",
            "Net_Amount"
        ]
    ].head()
)


# 15. Rating Category

df["Rating_Category"] = np.select(
    [
        df["Rating"].between(1, 2),
        df["Rating"] == 3,
        df["Rating"] == 4,
        df["Rating"] == 5
    ],
    [
        "Poor",
        "Average",
        "Good",
        "Excellent"
    ],
    default="Unknown"
)

print("\nRating Categories:")
print(df["Rating_Category"].value_counts())


# 16. Extract date information

df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Day"] = df["Order_Date"].dt.day
df["Day_of_Week"] = df["Order_Date"].dt.day_name()

print("\nDate features:")
print(
    df[
        [
            "Order_Date",
            "Year",
            "Month",
            "Day",
            "Day_of_Week"
        ]
    ].head()
)


# =========================================================
# PART D — BUSINESS ANALYSIS
# =========================================================

# 17. Total revenue

total_revenue = df["Net_Amount"].sum()

print("\n17. Total Revenue:")
print(round(total_revenue, 2))


# 18. Total number of orders

total_orders = df["Order_ID"].nunique()

print("\n18. Total Orders:")
print(total_orders)


# 19. Average Order Value

average_order_value = df["Net_Amount"].mean()

print("\n19. Average Order Value:")
print(round(average_order_value, 2))


# 20. Total revenue by product category

revenue_by_category = (
    df.groupby("Product_Category")["Net_Amount"]
      .sum()
      .sort_values(ascending=False)
)

print("\n20. Revenue by Product Category:")
print(revenue_by_category.round(2))


# 21. Top 10 products by revenue

top_10_products = (
    df.groupby("Product")["Net_Amount"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print("\n21. Top 10 Products by Revenue:")
print(top_10_products.round(2))


# 22. City generating highest revenue

revenue_by_city = (
    df.groupby("City")["Net_Amount"]
      .sum()
      .sort_values(ascending=False)
)

highest_revenue_city = revenue_by_city.idxmax()

print("\n22. Highest Revenue City:")
print(highest_revenue_city)

print("Revenue:")
print(round(revenue_by_city.max(), 2))


# 23. Most frequently used payment method

most_used_payment = df["Payment_Method"].mode()[0]

print("\n23. Most Frequently Used Payment Method:")
print(most_used_payment)


# 24. Average rating for each product category

average_rating_category = (
    df.groupby("Product_Category")["Rating"]
      .mean()
      .sort_values(ascending=False)
)

print("\n24. Average Rating by Product Category:")
print(average_rating_category.round(2))


# 25. Monthly revenue

monthly_revenue = (
    df.groupby(
        df["Order_Date"].dt.to_period("M")
    )["Net_Amount"]
    .sum()
)

print("\n25. Monthly Revenue:")
print(monthly_revenue.round(2))


# 26. Highest revenue month

highest_revenue_month = monthly_revenue.idxmax()
highest_month_revenue = monthly_revenue.max()

print("\n26. Highest Revenue Month:")
print(highest_revenue_month)

print("Revenue:")
print(round(highest_month_revenue, 2))


# 27. Top 10 customers by total spending

top_10_customers = (
    df.groupby("Customer_ID")["Net_Amount"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print("\n27. Top 10 Customers:")
print(top_10_customers.round(2))


# 28. Compare revenue generated by different cities

city_comparison = (
    df.groupby("City")["Net_Amount"]
      .sum()
      .sort_values(ascending=False)
)

print("\n28. Revenue by City:")
print(city_comparison.round(2))


# 29. Product category with highest average order value

category_aov = (
    df.groupby("Product_Category")["Net_Amount"]
      .mean()
      .sort_values(ascending=False)
)

highest_aov_category = category_aov.idxmax()

print("\n29. Highest Average Order Value Category:")
print(highest_aov_category)

print(
    "Average Order Value:",
    round(category_aov.max(), 2)
)

Dataset loaded successfully

First 5 records:
  Order_ID Customer_ID       City Product_Category    Product  Quantity  \
0  O014846      C00722      Delhi  Home Appliances      Mixer         2   
1  O014121      C01623       Pune            Books      Novel         1   
2  O014861      C01543  Hyderabad  Home Appliances      Mixer         2   
3  O000885      C02287     Mumbai            Books  Biography         5   
4  O000412      C00051     Mumbai           Beauty    Shampoo         5   

   Unit_Price  Discount Payment_Method                     Order_Date  Rating  
0    10042.08        10     Debit Card  2026-07-25 02:03:50.895393024     4.0  
1    34791.00        15     Debit Card  2026-06-27 05:51:34.526301752     1.0  
2    15693.00        15    Net Banking  2026-07-25 15:53:20.613374224     4.0  
3    72700.86        20     Debit Card  2025-02-03 22:44:55.379691979     3.0  
4    89712.89        25            UPI  2025-01-16 18:48:10.272684845     5.0  

Dataset dimensions:
(1